# RNA3D Phase 1 — DRfold2 structural evaluation vs NuFold

**Purpose.** Evaluate the ten already-completed DRfold2 PDB predictions using the *same native residue mapping, RNA-specific US-align options and normalization* as the archived NuFold single-sequence pilot, then produce per-target paired scores.

**This notebook DOES NOT run model inference, train models, modify GitHub, or overwrite existing prediction files.** It runs on a CPU-only Colab runtime. It reads existing artifacts from Google Drive and creates an independent **new, immutable** evaluation folder under `RNA3D_Benchmark/DRfold2/runs/drfold2_pilot10_20261110_v1/evaluations/` only after all checks pass.

**Scoring definition:** `USalign <native-observed-position-trimmed-prediction> <native_eval.pdb> -mol RNA -TMscore 1` using **US-align tag 20260527**. Take TM-score *normalized by Structure_2 (native)* and RMSD over aligned residues. The existing NuFold evaluation uses this exact command. A calibration step must reproduce archived NuFold rank-1 scores before any DRfold2 scoring is accepted.

⚠️ Native structures are sometimes incomplete. For example, `8ubd_H` has only 13 experimental residues out of 36. We report native coordinate availability separately from alignment coverage, and provide a >=80% observed-coordinate subset. Scores from this small pilot are exploratory, not the final template-sparse benchmark.

Run cells **top to bottom**. If a validation fails, fix the input/provenance rather than bypassing its assertion.

## 1. Mount the correct Google Drive
Use the Google account that contains the original `RNA3D_Benchmark` folder (the same Drive used for DRfold2 inference).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Immutable experiment settings and prerequisite paths

In [ ]:
from pathlib import Path
import os, csv, json, re, sys, shutil, subprocess, hashlib, math, time
from datetime import datetime, timezone
from statistics import mean, median

ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
NUFOLD_RUN = ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
DRFOLD2_RUN = ROOT/'DRfold2/runs/drfold2_pilot10_20261110_v1'
NATIVE = ROOT/'native/pilot_v1/native_eval'

TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B',
           '9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']

# This GitHub commit is an immutable read-only source for the original NuFold mapping functions.
PROJECT_URL = 'https://github.com/ABDULSALAM589/rna3d-template-sparse-benchmark.git'
PROJECT_COMMIT = 'c79a78b0a35be00a31aadeb04a6ebec5f2a0004f'
PROJECT = Path('/content/rna3d-eval-code')

# Same upstream version as the NuFold archived US-align evaluation.
USALIGN_TAG = '20260527'
USALIGN_SRC = Path('/content/USalign-rna3d-eval')
USALIGN = USALIGN_SRC/'USalign'

# New directory ONLY. Never use an existing evaluation ID without inspecting it.
EVALUATION_ID = 'drfold2_vs_nufold_usalign20260527_20261111_v1'
EVALUATION_DIR = DRFOLD2_RUN/'evaluations'/EVALUATION_ID
STAGE = Path('/content/rna3d_drfold2_evaluation_stage')

for path in [ROOT, NUFOLD_RUN, DRFOLD2_RUN, NATIVE]:
    assert path.is_dir(), f'Missing required directory: {path}'
assert len(TARGETS)==len(set(TARGETS))==10
print('DRfold2 run:', DRFOLD2_RUN)
print('NuFold baseline:', NUFOLD_RUN)
print('Native references:', NATIVE)
print('Planned new evaluation folder:', EVALUATION_DIR)
print('GPU is NOT required for scoring.')


## 3. Fetch read-only evaluation code from the pinned benchmark repo
The helper functions are imported from `scripts/nufold/common.py`; this notebook does not change or commit repository files.

In [ ]:
def checked(command, cwd=None):
    print('$', ' '.join(map(str, command)))
    subprocess.run(list(map(str,command)),cwd=cwd,check=True)

if not PROJECT.exists():
    checked(['git','clone','--filter=blob:none',PROJECT_URL,PROJECT])
checked(['git','-C',PROJECT,'checkout','--detach',PROJECT_COMMIT])
actual_commit=subprocess.check_output(['git','-C',PROJECT,'rev-parse','HEAD'],text=True).strip()
assert actual_commit==PROJECT_COMMIT, f'Project code changed: {actual_commit}'

sys.path.insert(0,str(PROJECT/'scripts/nufold'))
from common import sha256_file, pdb_residues, validate_ranked_pdb, read_native_positions, write_mapped_prediction
print('Pinned mapping implementation:', actual_commit)


## 4. Install the same US-align release used for NuFold evaluation
Compiles a lightweight CPU-only aligner. The official tag is `20260527`, not the moving `master` branch. Reference: https://github.com/pylelab/USalign

In [ ]:
if not USALIGN_SRC.exists():
    checked(['git','clone','--depth','1','--branch',USALIGN_TAG,
             'https://github.com/pylelab/USalign.git',USALIGN_SRC])
tag_head=subprocess.check_output(['git','-C',USALIGN_SRC,'rev-parse','HEAD'],text=True).strip()
tag_commit=subprocess.check_output(['git','-C',USALIGN_SRC,'rev-parse',f'refs/tags/{USALIGN_TAG}^{{commit}}'],text=True).strip()
assert tag_head==tag_commit, 'US-align source not at pinned tag'
if not USALIGN.exists():
    checked(['make'],cwd=USALIGN_SRC)
assert USALIGN.is_file() and os.access(USALIGN,os.X_OK), 'USalign executable not found'
print('US-align source commit:',tag_head)
print('US-align executable SHA256:',sha256_file(USALIGN))


## 5. Preflight all 10 targets and all frozen artifacts
Hashes must agree with each target's existing `complete.json`. We do not rely on folder existence alone.

In [ ]:
def fasta_sequence(path):
    lines=[x.strip() for x in path.read_text().splitlines() if x.strip()]
    assert lines and lines[0].startswith('>')
    assert sum(x.startswith('>') for x in lines)==1, f'Not single-record FASTA: {path}'
    seq=''.join(x for x in lines if not x.startswith('>')).upper()
    assert seq and set(seq)<=set('ACGU'),f'Invalid RNA sequence in {path}'
    return seq

nu_scores_path=NUFOLD_RUN/'analysis/rank1_results.csv'
assert nu_scores_path.is_file(), f'NuFold rank-1 scores missing: {nu_scores_path}'
with nu_scores_path.open(newline='') as f:
    nu_rows=list(csv.DictReader(f))
nu_scores={r['target_id']:r for r in nu_rows if r['rank']=='1' and r['status']=='scored'}
assert set(nu_scores)==set(TARGETS),f'NuFold scores missing: {set(TARGETS)-set(nu_scores)}'

inputs=[]
for t in TARGETS:
    fastas=list((NUFOLD_RUN/'inputs'/t).glob('*.fasta'))
    assert len(fastas)==1, f'{t}: expected exactly one FASTA'
    fasta=fastas[0]; seq=fasta_sequence(fasta)
    native=NATIVE/f'{t}_native_eval.pdb'
    residue_map=NATIVE/f'{t}_residue_map.csv'
    pred=DRFOLD2_RUN/t/'model_1.pdb'
    marker=DRFOLD2_RUN/t/'complete.json'
    assert all(p.is_file() for p in [native,residue_map,pred,marker]),f'{t}: incomplete required files'
    meta=json.loads(marker.read_text())
    assert meta['status']=='complete' and meta['target_id']==t, f'{t}: invalid marker'
    assert meta['model']=='DRfold2' and meta['model_commit']=='3667d6ac44c5b14cff94c36e0371224acb2cdae5'
    assert sha256_file(pred)==meta['pdb_sha256'],f'{t}: prediction SHA256 mismatch'
    assert sha256_file(fasta)==meta['fasta_sha256'],f'{t}: FASTA SHA256 mismatch'
    assert sha256_file(native)==meta['native_sha256'],f'{t}: native SHA256 mismatch'
    pred_details=validate_ranked_pdb(pred,len(seq))
    assert pred_details['sequence']==seq,f'{t}: DRfold2 prediction sequence differs from input'
    positions=read_native_positions(residue_map)
    assert positions==sorted(set(positions)),f'{t}: invalid observed residue positions'
    assert positions and min(positions)>=1 and max(positions)<=len(seq)
    native_res=pdb_residues(native)
    assert len(native_res)==len(positions),f'{t}: native structure count vs map mismatch'
    assert ''.join(r['one'] for r in native_res)==''.join(seq[i-1] for i in positions),f'{t}: native sequence/map mismatch'
    inputs.append({'target_id':t,'sequence':seq,'fasta':fasta,'pred':pred,'native':native,
                   'residue_map':residue_map,'positions':positions,'complete':meta,
                   'length':len(seq),'native_residues':len(positions),
                   'native_observed_fraction':len(positions)/len(seq)})
print('PRECHECK PASS: DRfold2 predictions, SHA256 hashes, input sequences and native maps: 10/10')
for q in inputs:
    print(f"{q['target_id']:9s}: predicted {q['length']:2d} nt; native {q['native_residues']:2d}/{q['length']:2d} ({q['native_observed_fraction']:.1%})")


## 6. US-align score parser + golden sanity test
The previous NuFold helper's parser was built for an older text format. The archived US-align v20260527 logs use `Aligned length=` and `TM-score=... normalized by length of Structure_2`. This parser supports both formats and insists that the **native-normalized** score is selected.

In [ ]:
def parse_scoring(stdout: str, native_count: int):
    # Modern US-align v20260527 output (the format archived with NuFold pilot).
    aligned=re.search(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([\d.eE+-]+)',stdout,re.I)
    if aligned:
        common=int(aligned.group(1))
        rmsd=float(aligned.group(2))
    else:
        common_m=re.search(r'Number of residues in common\s*=\s*(\d+)',stdout,re.I)
        rmsd_m=re.search(r'RMSD of the common residues\s*=\s*([\d.eE+-]+)',stdout,re.I)
        if not (common_m and rmsd_m):
            raise ValueError('US-align aligned length/RMSD not found in output')
        common=int(common_m.group(1));rmsd=float(rmsd_m.group(1))
    # With -TMscore 1 structure_2 is always the experimental/native PDB.
    ref_tm=re.findall(r'TM-score\s*=\s*([\d.eE+-]+)\s*\(normalized by length of Structure_2\b[^)]*\)',stdout,re.I)
    if not ref_tm:
        raise ValueError('Reference-normalized (Structure_2) TM-score not found')
    tm=float(ref_tm[-1])
    assert common>0 and common<=native_count,(common,native_count)
    assert math.isfinite(tm) and 0<=tm<=1.00001
    assert math.isfinite(rmsd) and rmsd>=0
    return {'alignment_residues':common,'alignment_coverage':common/native_count,
            'tm_score':tm,'rmsd_angstrom':rmsd}

SAMPLE_ARCHIVED_LOG="""Aligned length= 13, RMSD=  11.52, Seq_ID=n_identical/n_aligned= 1.000
TM-score= 0.24331 (normalized by length of Structure_1: L=13, d0=0.40)
TM-score= 0.24331 (normalized by length of Structure_2: L=13, d0=0.40)"""
assert parse_scoring(SAMPLE_ARCHIVED_LOG,13)=={'alignment_residues':13,'alignment_coverage':1.,'tm_score':0.24331,'rmsd_angstrom':11.52}

def score_mapped(mapped,native):
    command=[str(USALIGN),str(mapped),str(native),'-mol','RNA','-TMscore','1']
    p=subprocess.run(command,capture_output=True,text=True)
    log=p.stdout+('\n--- STDERR ---\n'+p.stderr if p.stderr else '')
    if p.returncode:
        raise RuntimeError(f'US-align failed with code {p.returncode}: {log[-800:]}')
    return log
print('Scoring parser unit test: PASS')


## 7. Calibration against archived NuFold scores (required)
Before scoring DRfold2, rerun the *exact* NuFold native-trimmed rank-1 PDBs through this freshly compiled US-align binary. We require every result to match the archived NuFold score within normal text rounding. A mismatch means **stop and debug the scorer**, not that DRfold2 is inaccurate.

In [ ]:
calibration=[]
for q in inputs:
    t=q['target_id']
    mapped=NUFOLD_RUN/'evaluations/pilot10_native_eval_v2'/t/f'{t}_rank_1_mapped.pdb'
    assert mapped.is_file(), f'{t}: NuFold mapped calibration PDB missing'
    output=score_mapped(mapped,q['native'])
    actual=parse_scoring(output,q['native_residues'])
    expected=nu_scores[t]
    tm_diff=abs(actual['tm_score']-float(expected['tm_score']))
    rmsd_diff=abs(actual['rmsd_angstrom']-float(expected['rmsd_angstrom']))
    assert tm_diff<=0.00006 and rmsd_diff<=0.06, (t, actual, expected, tm_diff, rmsd_diff)
    assert actual['alignment_residues']==int(expected['alignment_residues'])
    calibration.append({'target_id':t,'tm_diff':tm_diff,'rmsd_diff_angstrom':rmsd_diff})
print('GOLDEN CALIBRATION PASS: 10/10 archived NuFold TM-scores/RMSDs reproduced.')
print('US-align release:',USALIGN_TAG)
print('Max absolute TM-score discrepancy:',max(q['tm_diff'] for q in calibration))


## 8. Evaluate DRfold2 predictions — local staging only
All mapped PDBs and US-align logs are generated **locally** under `/content`; inputs in Drive are never edited. No model inference is run.

In [ ]:
assert len(calibration)==10, 'First complete NuFold golden calibration'
if STAGE.exists():
    raise FileExistsError(f'Existing local staging folder {STAGE}. Inspect first; do not automatically overwrite.')
STAGE.mkdir(parents=True)
(STAGE/'mapped').mkdir()
(STAGE/'logs').mkdir()
results=[]
for q in inputs:
    t=q['target_id']
    mapped=STAGE/'mapped'/f'{t}_mapped.pdb'
    kept=write_mapped_prediction(q['pred'],mapped,q['sequence'],q['positions'])
    assert kept==q['native_residues']
    text=score_mapped(mapped,q['native'])
    (STAGE/'logs'/f'{t}_usalign.txt').write_text(text)
    s=parse_scoring(text,q['native_residues'])
    assert s['alignment_residues']==q['native_residues'],f'{t}: mapped native residues not fully aligned; investigate'
    row={'target_id':t,'status':'scored','input_length':q['length'],
         'native_residues':q['native_residues'],
         'native_observed_fraction':q['native_observed_fraction'],
         'alignment_residues':s['alignment_residues'],
         'alignment_coverage':s['alignment_coverage'],
         'tm_score':s['tm_score'],'rmsd_angstrom':s['rmsd_angstrom'],
         'inference_runtime_seconds':q['complete']['seconds'],
         'prediction_sha256':q['complete']['pdb_sha256'],
         'model_commit':q['complete']['model_commit']}
    results.append(row)
    print(f"{t:9s}: TM {s['tm_score']:.5f} | RMSD {s['rmsd_angstrom']:.2f} Å | native {q['native_observed_fraction']:.0%}")
assert len(results)==10 and all(x['status']=='scored' for x in results)
print('DRfold2 structural scoring: 10/10 PASS (local staging; not saved to Drive yet)')


## 9. Create paired NuFold vs DRfold2 comparison and clearly separated aggregates
A *higher* ΔTM-score (`DRfold2 − NuFold`) favors DRfold2. A *lower* ΔRMSD favors DRfold2. Both methods use a model-default selected output, but their internal candidate-generation budgets differ; this is a **pilot**, not a definitive head-to-head performance ranking.

In [ ]:
paired=[]
for x in results:
    t=x['target_id'];n=nu_scores[t]
    a=float(n['tm_score']);b=float(n['rmsd_angstrom'])
    paired.append({'target_id':t,
                   'input_length':x['input_length'],
                   'native_residues':x['native_residues'],
                   'native_observed_fraction':x['native_observed_fraction'],
                   'nufold_rank1_tm_score':a,'drfold2_tm_score':x['tm_score'],
                   'delta_tm_drfold2_minus_nufold':x['tm_score']-a,
                   'nufold_rank1_rmsd_angstrom':b,'drfold2_rmsd_angstrom':x['rmsd_angstrom'],
                   'delta_rmsd_drfold2_minus_nufold':x['rmsd_angstrom']-b,
                   'native_alignment_coverage':x['alignment_coverage']})

def aggregate(subset):
    return {'n':len(subset),
            'mean_nufold_tm':mean(r['nufold_rank1_tm_score'] for r in subset),
            'mean_drfold2_tm':mean(r['drfold2_tm_score'] for r in subset),
            'mean_paired_delta_tm':mean(r['delta_tm_drfold2_minus_nufold'] for r in subset),
            'median_paired_delta_tm':median(r['delta_tm_drfold2_minus_nufold'] for r in subset),
            'mean_nufold_rmsd_angstrom':mean(r['nufold_rank1_rmsd_angstrom'] for r in subset),
            'mean_drfold2_rmsd_angstrom':mean(r['drfold2_rmsd_angstrom'] for r in subset),
            'mean_paired_delta_rmsd_angstrom':mean(r['delta_rmsd_drfold2_minus_nufold'] for r in subset)}

high_coverage=[x for x in paired if x['native_observed_fraction']>=0.80]
assert high_coverage and len(high_coverage)<len(paired)
summary={'all_ten_exploratory':aggregate(paired),
         'native_observed_fraction_ge_0p80':aggregate(high_coverage),
         'caveat':'Pilot only; native-coordinate coverage can be low and model candidate-selection budgets differ.'}
print('ALL 10 exploratory:',summary['all_ten_exploratory'])
print('HIGH NATIVE COVERAGE (>=80%):',summary['native_observed_fraction_ge_0p80'])
print('\nPer-target comparison:')
for r in paired:
    print(f"{r['target_id']:9s} NuFold {r['nufold_rank1_tm_score']:.5f}, DRfold2 {r['drfold2_tm_score']:.5f}, ΔTM {r['delta_tm_drfold2_minus_nufold']:+.5f}, native available {r['native_observed_fraction']:.1%}")


## 10. Save an independent, immutable evaluation folder to Google Drive
This cell **does not modify GitHub** and does not overwrite the old NuFold/DRfold2 experiment. It copies only small new evaluation artifacts and writes a completion marker last. If an evaluation directory already exists, it refuses to overwrite it.

In [ ]:
def write_csv(path, rows):
    assert rows
    with path.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]),lineterminator='\n')
        writer.writeheader();writer.writerows(rows)

write_csv(STAGE/'drfold2_scores.csv',results)
write_csv(STAGE/'nufold_vs_drfold2_paired.csv',paired)
write_csv(STAGE/'nufold_score_calibration.csv',calibration)
provenance={'evaluation_id':EVALUATION_ID,
            'created_utc':datetime.now(timezone.utc).isoformat(),
            'run_id':'drfold2_pilot10_20261110_v1',
            'nufold_run_id':'pilot10_single_20261018T203832Z',
            'project_source_commit':actual_commit,
            'usalign_tag':USALIGN_TAG,
            'usalign_source_commit':tag_head,
            'usalign_binary_sha256':sha256_file(USALIGN),
            'usalign_command':'USalign <trimmed_prediction> <native> -mol RNA -TMscore 1',
            'tm_score_normalization':'Structure_2 (experimental native)',
            'reference_policy':'Observed native residues only; output requires explicit target-position mapping',
            'target_count':len(TARGETS),
            'nufold_golden_calibration_count':len(calibration),
            'summary':summary,
            'note':'Model inference was not repeated. These are scores for existing outputs, not a final benchmark.'}
(STAGE/'summary.json').write_text(json.dumps(provenance,indent=2)+'\n')
assert not EVALUATION_DIR.exists(), (f'Evaluation output already exists: {EVALUATION_DIR}; '
                                     'inspect it or choose a NEW EVALUATION_ID; never overwrite')
EVALUATION_DIR.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(STAGE,EVALUATION_DIR)
for name in ['drfold2_scores.csv','nufold_vs_drfold2_paired.csv','nufold_score_calibration.csv','summary.json']:
    assert sha256_file(EVALUATION_DIR/name)==sha256_file(STAGE/name),f'Copy verification failed: {name}'
# End-of-copy sentinel so an interrupted Drive copy is never mistaken for complete.
(EVALUATION_DIR/'FINALIZED.json').write_text(json.dumps({
    'status':'complete','evaluation_id':EVALUATION_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'files':{name:sha256_file(EVALUATION_DIR/name) for name in
             ['drfold2_scores.csv','nufold_vs_drfold2_paired.csv','nufold_score_calibration.csv','summary.json']}},indent=2)+'\n')
print('SAVED, VERIFIED:', EVALUATION_DIR)
print('Paired comparison CSV:',EVALUATION_DIR/'nufold_vs_drfold2_paired.csv')
print('IMPORTANT: No GitHub commit/push was made.')


## 11. Read-only verification in a fresh Colab session
If the runtime disconnects *after* `FINALIZED.json` is written, these saved files survive in Drive; this cell can be executed independently after mounting Drive and setting `EVALUATION_DIR` to the path printed above.

In [ ]:
final=EVALUATION_DIR/'FINALIZED.json'
assert final.is_file(), 'Finalized evaluation not found. Check whether saving completed.'
marker=json.loads(final.read_text())
assert marker['status']=='complete'
for name,digest in marker['files'].items():
    assert sha256_file(EVALUATION_DIR/name)==digest, f'Corrupted output: {name}'
print('EVALUATION ARTIFACT VERIFICATION PASS')
print('All outputs:',*[p.name for p in EVALUATION_DIR.iterdir()],sep='\n - ')
